# 02_block_cam_sensitivity_mel_nv

Qualitative block sensitivity check for MEL vs NV CAMs.

Goal:
- Compare where heatmaps focus across transformer blocks.
- Use this as an internal visual analysis notebook.
- Notebook 03 can then do quantitative block analysis.

Default blocks:
`[-1, -2, -4, -6, -8, -10, -12]`

Output design:
- Generate panels for CLS and GAP separately.
- Build one PDF per model.
- Each PDF page = one image.
- Each page has one row per block.
- Columns: RGB lesion outline, Grad CAM target, Diff CAM, Finer CAM.


In [1]:
from pathlib import Path
import json
import subprocess
import shlex
import pandas as pd

# =============================================================================
# 1. MAIN PARAMETERS
# =============================================================================

# Notebook location: notebooks/mel_nv/02_block_cam_sensitivity_mel_nv.ipynb
# REPO_ROOT = Path("../..").resolve()  # local notebook
REPO_ROOT = Path("..").resolve() / "master-thesis"  # UBELIX notebook

HAM_ROOT = REPO_ROOT / "data" / "HAM10000"
MEL_NV_ROOT = HAM_ROOT / "mel_nv"

IMG_DIR = HAM_ROOT
MASK_ROOT = HAM_ROOT

# We are no longer running all_test. We are running a curated clinician review set.
SAMPLE_MODE = "clinician_review"

# Show only the most relevant blocks.
TARGET_BLOCK_INDICES = [-1, -4, -10]

DRY_RUN = False
RUN_PANEL_GENERATION = True
BUILD_PDFS = True

# This notebook should only create CAMs and PDFs now.
RUN_ALIGNMENT_METRICS = False

# Panel columns
PANEL_ITEMS = "rgb_gt_mask,gradcam_a,map_diff,finercam"
PANEL_SUFFIX = PANEL_ITEMS.replace(",", "_")

# Finer-CAM comparison strength
ALPHA = 0.8

# Paths
CLEAN_CSV = MEL_NV_ROOT / "ham_mel_nv_clean.csv"

CURATION_CSV = (
    REPO_ROOT
    / "outputs"
    / "mel_nv"
    / "feature_space_difficult_cases_last_block"
    / "clinician_curation_candidates"
    / "clinician_curation_candidates_combined_cls_gap.csv"
)

CLS_CKPT = REPO_ROOT / "external" / "checkpoints4" / "checkpoint-best-cls.pth"
GAP_CKPT = REPO_ROOT / "external" / "checkpoints4" / "checkpoint-best-gap.pth"

GT_COL = "gt_label"

CLASS_ARGS = ["--class_names", "MEL,NV"]
COMPARE_ARGS = [
    "--compare_mode", "gt_pair",
    "--A", "MEL",
    "--B", "NV",
    "--topk_compare", "1",
]

SCENARIOS = [
    {
        "name": "CLS HA 0.5",
        "short_name": "cls_ha05",
        "checkpoint": CLS_CKPT,
        "checkpoint_model_type": "panderm",
        "pooling": "cls",
    },
    {
        "name": "GAP HA 0.25",
        "short_name": "gap_ha025",
        "checkpoint": GAP_CKPT,
        "checkpoint_model_type": "panderm",
        "pooling": "mean",
    },
]

OUT_ROOT = REPO_ROOT / "outputs" / "mel_nv" / "clinician_review_cams"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("OUT_ROOT:", OUT_ROOT)
print("CLEAN_CSV exists:", CLEAN_CSV.exists(), CLEAN_CSV)
print("CURATION_CSV exists:", CURATION_CSV.exists(), CURATION_CSV)
print("CLS_CKPT exists:", CLS_CKPT.exists(), CLS_CKPT)
print("GAP_CKPT exists:", GAP_CKPT.exists(), GAP_CKPT)
print("Blocks:", TARGET_BLOCK_INDICES)

for scenario in SCENARIOS:
    print("\n", scenario["name"])
    print("  checkpoint:", scenario["checkpoint"])
    print("  pooling:", scenario["pooling"])

REPO_ROOT: /storage/homefs/cn21m021/projects/master-thesis
OUT_ROOT: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/clinician_review_cams
CLEAN_CSV exists: True /storage/homefs/cn21m021/projects/master-thesis/data/HAM10000/mel_nv/ham_mel_nv_clean.csv
CURATION_CSV exists: True /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/feature_space_difficult_cases_last_block/clinician_curation_candidates/clinician_curation_candidates_combined_cls_gap.csv
CLS_CKPT exists: True /storage/homefs/cn21m021/projects/master-thesis/external/checkpoints4/checkpoint-best-cls.pth
GAP_CKPT exists: True /storage/homefs/cn21m021/projects/master-thesis/external/checkpoints4/checkpoint-best-gap.pth
Blocks: [-1, -4, -10]

 CLS HA 0.5
  checkpoint: /storage/homefs/cn21m021/projects/master-thesis/external/checkpoints4/checkpoint-best-cls.pth
  pooling: cls

 GAP HA 0.25
  checkpoint: /storage/homefs/cn21m021/projects/master-thesis/external/checkpoints4/checkpoint-best-gap.pth
  pooling:

## 2. Build active CSV


In [2]:
def safe_name(name: str) -> str:
    out = str(name).lower()
    for ch in [" ", "/", "\\", ":", ";", ",", "(", ")", "[", "]", "{", "}", "+", "."]:
        out = out.replace(ch, "_")
    while "__" in out:
        out = out.replace("__", "_")
    return out.strip("_")


def prepare_review_csvs():
    """
    Creates three clinician review CSVs:
    1. typical MEL
    2. typical NV
    3. difficult cases

    Important:
    We deduplicate by image_id because the combined curation CSV contains rows from both CLS and GAP.
    """
    csv_out_dir = OUT_ROOT / "csv"
    csv_out_dir.mkdir(parents=True, exist_ok=True)

    clean_df = pd.read_csv(CLEAN_CSV, low_memory=False)
    cur = pd.read_csv(CURATION_CSV, low_memory=False)

    print("Curation shape:", cur.shape)
    print("Curation groups:")
    display(cur["curation_group"].value_counts())

    # Deduplicate within each group by image_id.
    # Prefer cases selected by both models where possible.
    if "model_short" in cur.columns:
        model_count = cur.groupby("image_id")["model_short"].nunique().rename("n_models_selected")
        cur = cur.merge(model_count, on="image_id", how="left")
    else:
        cur["n_models_selected"] = 1

    # 1. Typical MEL: nearest MEL centroid, unique image IDs
    typical_mel_ids = (
        cur[cur["curation_group"].eq("nearest_mel_centroid")]
        .sort_values(["n_models_selected", "mel_probability"], ascending=[False, False])
        .drop_duplicates("image_id")
        .head(10)["image_id"]
        .tolist()
    )

    # 2. Typical NV: nearest NV centroid, unique image IDs
    typical_nv_ids = (
        cur[cur["curation_group"].eq("nearest_nv_centroid")]
        .sort_values(["n_models_selected", "mel_probability"], ascending=[False, True])
        .drop_duplicates("image_id")
        .head(10)["image_id"]
        .tolist()
    )

    # 3. Difficult cases:
    # all FN MEL + top FP MEL, deduplicated.
    fn_ids = (
        cur[cur["curation_group"].eq("fn_mel_all")]
        .sort_values(["n_models_selected", "mel_probability"], ascending=[False, True])
        .drop_duplicates("image_id")["image_id"]
        .tolist()
    )

    fp_ids = (
        cur[cur["curation_group"].eq("fp_mel_top10")]
        .sort_values(["n_models_selected", "mel_probability"], ascending=[False, False])
        .drop_duplicates("image_id")
        .head(10)["image_id"]
        .tolist()
    )

    difficult_ids = list(dict.fromkeys(fn_ids + fp_ids))

    review_sets = {
        "typical_mel": typical_mel_ids,
        "typical_nv": typical_nv_ids,
        "difficult_cases": difficult_ids,
    }

    review_csvs = {}

    for review_name, image_ids in review_sets.items():
        df_review = clean_df[clean_df["image_id"].astype(str).isin([str(x) for x in image_ids])].copy()

        # Preserve the selected order.
        order_map = {str(img_id): i for i, img_id in enumerate(image_ids)}
        df_review["review_order"] = df_review["image_id"].astype(str).map(order_map)
        df_review["review_group"] = review_name
        df_review = df_review.sort_values("review_order").reset_index(drop=True)

        out_csv = csv_out_dir / f"clinician_review_{review_name}.csv"
        df_review.to_csv(out_csv, index=False)

        review_csvs[review_name] = {
            "csv": out_csv,
            "df": df_review,
            "num_samples": len(df_review),
        }

        print("\n" + "=" * 80)
        print(review_name)
        print("CSV:", out_csv)
        print("n:", len(df_review))
        display(df_review[["review_group", "review_order", "image_id", "gt_label", "image_rel_path", "mask_rel_path"]].head(20))

    return review_csvs


REVIEW_CSVS = prepare_review_csvs()

Curation shape: (65, 30)
Curation groups:


curation_group
nearest_mel_centroid    20
nearest_nv_centroid     20
fp_mel_top10            20
fn_mel_all               5
Name: count, dtype: int64


typical_mel
CSV: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/clinician_review_cams/csv/clinician_review_typical_mel.csv
n: 10


,review_group,review_order,image_id,gt_label,image_rel_path,mask_rel_path
0,typical_mel,0,ISIC_0030246,MEL,images/ISIC_0030246.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
1,typical_mel,1,ISIC_0031408,MEL,images/ISIC_0031408.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
2,typical_mel,2,ISIC_0025132,MEL,images/ISIC_0025132.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
3,typical_mel,3,ISIC_0029089,MEL,images/ISIC_0029089.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
4,typical_mel,4,ISIC_0029698,MEL,images/ISIC_0029698.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
5,typical_mel,5,ISIC_0025414,MEL,images/ISIC_0025414.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
6,typical_mel,6,ISIC_0028897,MEL,images/ISIC_0028897.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
7,typical_mel,7,ISIC_0027420,MEL,images/ISIC_0027420.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
8,typical_mel,8,ISIC_0032244,MEL,images/ISIC_0032244.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
9,typical_mel,9,ISIC_0031642,MEL,images/ISIC_0031642.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...



typical_nv
CSV: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/clinician_review_cams/csv/clinician_review_typical_nv.csv
n: 10


,review_group,review_order,image_id,gt_label,image_rel_path,mask_rel_path
0,typical_nv,0,ISIC_0026181,NV,images/ISIC_0026181.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
1,typical_nv,1,ISIC_0032143,NV,images/ISIC_0032143.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
2,typical_nv,2,ISIC_0026582,NV,images/ISIC_0026582.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
3,typical_nv,3,ISIC_0031246,NV,images/ISIC_0031246.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
4,typical_nv,4,ISIC_0024738,NV,images/ISIC_0024738.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
5,typical_nv,5,ISIC_0025006,NV,images/ISIC_0025006.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
6,typical_nv,6,ISIC_0028535,NV,images/ISIC_0028535.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
7,typical_nv,7,ISIC_0030718,NV,images/ISIC_0030718.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
8,typical_nv,8,ISIC_0029599,NV,images/ISIC_0029599.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
9,typical_nv,9,ISIC_0031879,NV,images/ISIC_0031879.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...



difficult_cases
CSV: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/clinician_review_cams/csv/clinician_review_difficult_cases.csv
n: 14


,review_group,review_order,image_id,gt_label,image_rel_path,mask_rel_path
0,difficult_cases,0,ISIC_0028173,MEL,images/ISIC_0028173.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
1,difficult_cases,1,ISIC_0030552,MEL,images/ISIC_0030552.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
2,difficult_cases,2,ISIC_0024886,MEL,images/ISIC_0024886.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
3,difficult_cases,3,ISIC_0029013,MEL,images/ISIC_0029013.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
4,difficult_cases,4,ISIC_0028999,NV,images/ISIC_0028999.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
5,difficult_cases,5,ISIC_0026491,NV,images/ISIC_0026491.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
6,difficult_cases,6,ISIC_0024344,NV,images/ISIC_0024344.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
7,difficult_cases,7,ISIC_0026209,NV,images/ISIC_0026209.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
8,difficult_cases,8,ISIC_0031725,NV,images/ISIC_0031725.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...
9,difficult_cases,9,ISIC_0026584,NV,images/ISIC_0026584.jpg,../HAM10000_segmentations_lesion_tschandl/ISIC...


## 3. Generate CAM panels for all blocks

This calls `scripts.generate_finer_cam_panderm` once per model and block.

Output folder pattern:

`outputs/mel_nv/block_cam_sensitivity_<mode>/panels/<model>/block_<block>/`


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no cuda")


def run_command(cmd: list[str], dry_run: bool = False):
    print("\n" + "=" * 100)
    print(" ".join(shlex.quote(str(x)) for x in cmd))
    print("=" * 100)
    if dry_run:
        return
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)


def block_dir_name(block_index: int) -> str:
    return f"block_{block_index}".replace("-", "minus")


def generate_panels_for_review_sets():
    panel_root = OUT_ROOT / "panels"
    panel_root.mkdir(parents=True, exist_ok=True)

    for review_name, pack in REVIEW_CSVS.items():
        active_csv = pack["csv"]
        num_samples = pack["num_samples"]

        for scenario in SCENARIOS:
            for block_index in TARGET_BLOCK_INDICES:
                scenario_out_dir = (
                    panel_root
                    / review_name
                    / scenario["short_name"]
                    / block_dir_name(block_index)
                )
                scenario_out_dir.mkdir(parents=True, exist_ok=True)

                cmd = [
                    "python", "-m", "scripts.generate_finer_cam_panderm",
                    "--csv", str(active_csv),
                    "--image_col", "image_rel_path",
                    "--img_dir", str(IMG_DIR),
                    "--gt_col", GT_COL,
                    "--checkpoint", str(scenario["checkpoint"]),
                    "--checkpoint_model_type", scenario.get("checkpoint_model_type", "panderm"),
                    "--pooling", scenario["pooling"],
                    "--out_dir", str(scenario_out_dir),
                    "--num_samples", str(num_samples),
                    "--method", "finercam",
                    "--alpha", str(ALPHA),
                    "--panel_items", PANEL_ITEMS,
                    "--mask_root", str(MASK_ROOT),
                    "--mask_col", "mask_rel_path",
                    "--target_block_index", str(block_index),
                    "--clinician_labels",
                    "--model_display_name", f"{scenario['name']} | {review_name} | block {block_index}",
                    "--save_raw_cams",
                ]
                cmd += CLASS_ARGS
                cmd += COMPARE_ARGS

                print(f"\nGenerating: {review_name} | {scenario['name']} | block {block_index}")
                run_command(cmd, dry_run=DRY_RUN)


if RUN_PANEL_GENERATION:
    generate_panels_for_review_sets()
else:
    print("RUN_PANEL_GENERATION=False, skipping CAM/panel generation.")

## 4. Quantitative lesion alignment per block

This computes simple lesion alignment metrics from saved raw CAMs:

- `top10_inside`: fraction of top 10% hottest CAM pixels inside the lesion mask
- `pointing_game`: whether the hottest CAM pixel is inside the lesion mask
- `inside_mean`: mean CAM value inside lesion
- `outside_mean`: mean CAM value outside lesion

These metrics do not prove clinical correctness, but they help choose a model/block where the heatmap is at least lesion-oriented.

In [4]:
from PIL import Image, ImageDraw, ImageFont


def get_font(size: int, bold: bool = False):
    candidates = [
        "/System/Library/Fonts/Supplemental/Arial Bold.ttf" if bold else "/System/Library/Fonts/Supplemental/Arial.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf" if bold else "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    ]
    for path in candidates:
        if path and Path(path).exists():
            return ImageFont.truetype(path, size=size)
    return ImageFont.load_default()


FONT_TITLE = get_font(34, bold=True)
FONT_SUBTITLE = get_font(22, bold=False)
FONT_LABEL = get_font(22, bold=True)
FONT_SMALL = get_font(17, bold=False)


def image_id_to_stem(image_id_value: str) -> str:
    p = Path(str(image_id_value))
    stem = p.stem if p.suffix else p.name
    return stem.replace("/", "_").replace("\\", "_").replace(" ", "_")


def find_panel_png(review_name: str, scenario: dict, block_index: int, row: pd.Series) -> Path | None:
    out_dir = (
        OUT_ROOT
        / "panels"
        / review_name
        / scenario["short_name"]
        / block_dir_name(block_index)
    )

    candidates = []
    if "image_rel_path" in row and pd.notna(row["image_rel_path"]):
        candidates.append(image_id_to_stem(row["image_rel_path"]))
    if "image_id" in row and pd.notna(row["image_id"]):
        candidates.append(image_id_to_stem(row["image_id"]))
    if "image" in row and pd.notna(row["image"]):
        candidates.append(image_id_to_stem(row["image"]))

    for stem in dict.fromkeys(candidates):
        direct = out_dir / f"{stem}_{PANEL_SUFFIX}.png"
        if direct.exists():
            return direct

        matches = sorted(out_dir.glob(f"{stem}_*.png"))
        if matches:
            return matches[0]

    return None


def make_page_for_review_image(review_name: str, row: pd.Series) -> Image.Image:
    page_width = 2400
    margin = 45
    label_width = 260
    gap = 14
    title_h = 125

    available_panel_width = page_width - 2 * margin - label_width - gap

    loaded_rows = []

    for scenario in SCENARIOS:
        for block_index in TARGET_BLOCK_INDICES:
            panel_path = find_panel_png(review_name, scenario, block_index, row)

            if panel_path is None:
                loaded_rows.append((scenario, block_index, None, None))
                continue

            panel = Image.open(panel_path).convert("RGB")
            scale = available_panel_width / panel.width
            new_h = int(panel.height * scale)
            panel = panel.resize((available_panel_width, new_h), Image.Resampling.LANCZOS)

            loaded_rows.append((scenario, block_index, panel, panel_path))

    row_heights = [panel.height if panel is not None else 190 for _, _, panel, _ in loaded_rows]
    page_height = title_h + margin + sum(row_heights) + gap * (len(row_heights) - 1) + margin

    page = Image.new("RGB", (page_width, page_height), "white")
    draw = ImageDraw.Draw(page)

    image_id = row.get("image_id", row.get("image_rel_path", "unknown"))
    gt = row.get(GT_COL, row.get("gt_label", "unknown"))
    review_order = row.get("review_order", "")
    review_group = row.get("review_group", review_name)

    title = f"Clinician CAM review: {review_group}"
    subtitle = f"Case {review_order} | Image: {image_id} | Ground truth: {gt} | Blocks: {TARGET_BLOCK_INDICES}"

    draw.text((margin, 26), title, fill="black", font=FONT_TITLE)
    draw.text((margin, 78), subtitle, fill=(60, 60, 60), font=FONT_SUBTITLE)

    y = title_h

    for scenario, block_index, panel, panel_path in loaded_rows:
        row_h = panel.height if panel is not None else 190

        label_x = margin
        label_y = y + 20

        draw.text((label_x, label_y), scenario["short_name"], fill="black", font=FONT_LABEL)
        draw.text((label_x, label_y + 34), f"block {block_index}", fill=(80, 80, 80), font=FONT_SMALL)
        draw.text((label_x, label_y + 58), scenario["pooling"], fill=(80, 80, 80), font=FONT_SMALL)

        if panel is None:
            box_x = margin + label_width + gap
            box_y = y
            draw.rectangle(
                [box_x, box_y, box_x + available_panel_width, box_y + row_h],
                outline=(180, 180, 180),
                width=2,
            )
            draw.text(
                (box_x + 30, box_y + 65),
                "Missing panel PNG",
                fill=(160, 0, 0),
                font=FONT_LABEL,
            )
        else:
            page.paste(panel, (margin + label_width + gap, y))

        y += row_h + gap

    return page


def build_pdf_for_review_group(review_name: str, df_review: pd.DataFrame):
    pages = []

    for _, row in df_review.iterrows():
        pages.append(make_page_for_review_image(review_name, row))

    if not pages:
        raise RuntimeError(f"No pages generated for {review_name}")

    pdf_out = OUT_ROOT / f"clinician_review_{review_name}_cls_gap_blocks_minus1_minus4_minus10.pdf"
    pages[0].save(pdf_out, save_all=True, append_images=pages[1:], resolution=150.0)

    print("Saved PDF:", pdf_out)
    return pdf_out


PDF_OUTPUTS = []

if DRY_RUN:
    print("DRY_RUN=True, skipping PDF build.")
elif not BUILD_PDFS:
    print("BUILD_PDFS=False, skipping PDF build.")
else:
    for review_name, pack in REVIEW_CSVS.items():
        PDF_OUTPUTS.append(build_pdf_for_review_group(review_name, pack["df"]))

print("\nPDF outputs:")
for p in PDF_OUTPUTS:
    print(" ", p)

Saved PDF: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/clinician_review_cams/clinician_review_typical_mel_cls_gap_blocks_minus1_minus4_minus10.pdf
Saved PDF: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/clinician_review_cams/clinician_review_typical_nv_cls_gap_blocks_minus1_minus4_minus10.pdf
Saved PDF: /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/clinician_review_cams/clinician_review_difficult_cases_cls_gap_blocks_minus1_minus4_minus10.pdf

PDF outputs:
  /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/clinician_review_cams/clinician_review_typical_mel_cls_gap_blocks_minus1_minus4_minus10.pdf
  /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/clinician_review_cams/clinician_review_typical_nv_cls_gap_blocks_minus1_minus4_minus10.pdf
  /storage/homefs/cn21m021/projects/master-thesis/outputs/mel_nv/clinician_review_cams/clinician_review_difficult_cases_cls_gap_blocks_minus1_minus4_minus10.pdf
